# OOP Part 2 — Encapsulation & the `static` Keyword
### 25 Questions

**Important context before you start:** Python doesn't have TRUE private variables like Java/C++ do. Encapsulation in Python is based on CONVENTION and mild enforcement, not a hard wall. This surprises people coming from other languages — so we'll spend real time on exactly what Python does and doesn't protect.

**On "static":** Python has no `static` keyword. The equivalent ideas are `@staticmethod` and `@classmethod` decorators. We'll cover both, and be explicit about how they map to what "static" means in other languages.

Same rules: attempt first, run your own cell, then compare.

---

## Part A: Public Attributes (the default) (Q1-Q3)

**Q1** Define a `Customer` class with a PUBLIC attribute `self.balance` set in `__init__`. Create an instance and directly modify `.balance` from OUTSIDE the class (`c.balance = 99999`) — confirm Python allows this with zero restriction, even to a nonsensical value.

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
class Customer:
    def __init__(self, name, balance):
        self.name = name
        self.balance = balance

c = Customer("Alice", 1000)
c.balance = 99999
print(c.balance)
# Nothing stopped this — public attributes are fully open to direct modification.

**Q2** Explain in a comment why DIRECTLY exposing `.balance` like this can be risky in real code — think about what could go wrong if external code sets it to something invalid (e.g. a negative number, or a string).

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
# If any code anywhere can do c.balance = -500 or c.balance = "oops",
# nothing prevents invalid states. There's no validation, no logging of the
# change, and no way to run side-effect logic (like updating a "last_modified"
# timestamp) when balance changes. As the codebase grows, this makes bugs
# from invalid states much harder to trace back to their source.
print("See comment above")

**Q3** Create a `Customer` instance and use `vars(c)` (or `c.__dict__`) to see ALL of its attributes as a dict — this is how you can inspect any object's current state regardless of how "hidden" its attributes are meant to be.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
class Customer:
    def __init__(self, name, balance):
        self.name = name
        self.balance = balance

c = Customer("Alice", 1000)
print(vars(c))
print(c.__dict__)

## Part B: The Single Underscore Convention (`_protected`) (Q4-Q7)

**Q4** Define a `Customer` class with `self._balance` (single leading underscore) instead of `self.balance`. This is a CONVENTION meaning "internal use, please don't touch directly" — but Python does NOT enforce it. Confirm you can still access and modify `c._balance` from outside.

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
class Customer:
    def __init__(self, name, balance):
        self.name = name
        self._balance = balance

c = Customer("Alice", 1000)
print(c._balance)
c._balance = 5000
print(c._balance)
# Still fully accessible — the underscore is a signal to other developers, not a lock.

**Q5** Add a "getter" METHOD `get_balance(self)` that returns `self._balance`, and a "setter" method `set_balance(self, amount)` that only updates `self._balance` if `amount >= 0` (silently ignoring invalid values otherwise). Test with a valid and an invalid amount.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
class Customer:
    def __init__(self, name, balance):
        self.name = name
        self._balance = balance

    def get_balance(self):
        return self._balance

    def set_balance(self, amount):
        if amount >= 0:
            self._balance = amount

c = Customer("Alice", 1000)
c.set_balance(2000)
print(c.get_balance())
c.set_balance(-500)
print(c.get_balance())  # unchanged, invalid amount was rejected

**Q6** Rewrite Q5 using Python's `@property` decorator instead of manual getter/setter methods — define `balance` as a property with a getter, and a `@balance.setter` that validates `amount >= 0`. The key benefit: calling code now looks like `c.balance` (no parentheses) instead of `c.get_balance()`, but validation still runs.

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
class Customer:
    def __init__(self, name, balance):
        self.name = name
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, amount):
        if amount >= 0:
            self._balance = amount

c = Customer("Alice", 1000)
print(c.balance)
c.balance = 2000
print(c.balance)
c.balance = -500
print(c.balance)  # unchanged — rejected by the setter, but syntax looks like plain attribute access

**Q7** Using the `@property` version from Q6, try `c.balance = -500` again but this time add a `print("Rejected invalid balance")` inside the setter's failing branch so invalid attempts are at least VISIBLE instead of silently swallowed. Test it.

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
class Customer:
    def __init__(self, name, balance):
        self.name = name
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, amount):
        if amount >= 0:
            self._balance = amount
        else:
            print("Rejected invalid balance")

c = Customer("Alice", 1000)
c.balance = -500
print(c.balance)

## Part C: The Double Underscore "Name Mangling" (`__private`) (Q8-Q12)

**Q8** Define a `Customer` class with `self.__balance` (DOUBLE leading underscore, no trailing). Try to access `c.__balance` directly from outside the class — observe this raises `AttributeError`, unlike the single-underscore version.

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
class Customer:
    def __init__(self, name, balance):
        self.name = name
        self.__balance = balance

c = Customer("Alice", 1000)
try:
    print(c.__balance)
except AttributeError as e:
    print("Error:", e)

**Q9** The double underscore triggers "name mangling" — Python actually renames `__balance` internally to `_Customer__balance`. Confirm this by accessing `c._Customer__balance` directly (it works, proving this is obscurity, not true security).

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
class Customer:
    def __init__(self, name, balance):
        self.name = name
        self.__balance = balance

c = Customer("Alice", 1000)
print(c._Customer__balance)
# It's still accessible if you know the mangled name — Python has NO true
# private attributes. Name mangling exists mainly to avoid accidental name
# clashes in inheritance (covered next notebook), not to enforce security.

**Q10** Add a public method `get_balance(self)` INSIDE the class that returns `self.__balance` — this is the intended way to expose double-underscore attributes: through methods defined within the class, which CAN see the real name.

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
class Customer:
    def __init__(self, name, balance):
        self.name = name
        self.__balance = balance

    def get_balance(self):
        return self.__balance

c = Customer("Alice", 1000)
print(c.get_balance())

**Q11** Write a comment summarizing the THREE levels of Python attribute "privacy" you've now seen: `self.attr` (public), `self._attr` (protected — convention only), `self.__attr` (name-mangled — harder to access by accident, still not truly private). When would you use each?

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
# self.attr        -> Public: freely intended to be accessed/modified from outside.
#                       Use for simple data with no validation needs.
#
# self._attr        -> Protected (convention): signals "internal use, treat as
#                       implementation detail" but nothing stops external access.
#                       Common for internal helper attributes.
#
# self.__attr        -> Name-mangled: harder to access by accident from outside
#                       or in subclasses, reduces accidental collisions. Use
#                       sparingly, mainly when you specifically want to avoid
#                       subclass attribute name clashes (see inheritance notebook).
#
# In practice, most Python code uses public attributes + @property for validation
# (Part B), and reserves double-underscore for specific internal implementation
# details that should never be touched directly.
print("See comment above")

**Q12** Build a `BankAccount` class with `self.__balance` (double underscore), a `deposit(self, amount)` method requiring `amount > 0` (else raise `ValueError`), and a `withdraw(self, amount)` method requiring `amount <= self.__balance` (else raise `ValueError`). This demonstrates the REAL purpose of encapsulation: forcing all changes through validated methods. Test both a valid and invalid deposit/withdrawal.

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
class BankAccount:
    def __init__(self, balance=0):
        self.__balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self.__balance += amount

    def withdraw(self, amount):
        if amount > self.__balance:
            raise ValueError("Insufficient funds")
        self.__balance -= amount

    def get_balance(self):
        return self.__balance

acc = BankAccount(1000)
acc.deposit(500)
print(acc.get_balance())

try:
    acc.withdraw(10000)
except ValueError as e:
    print("Error:", e)

## Part D: `@staticmethod` — Python's "static" (Q13-Q18)

**Q13** Define a class `CreditUtils` with a `@staticmethod` method `is_valid_score(score)` that returns `True` if `300 <= score <= 850`. Notice it does NOT take `self` — call it directly on the CLASS: `CreditUtils.is_valid_score(720)`, with no instance needed.

In [ ]:
# YOUR CODE HERE


**Solution 13**

In [ ]:
class CreditUtils:
    @staticmethod
    def is_valid_score(score):
        return 300 <= score <= 850

print(CreditUtils.is_valid_score(720))
print(CreditUtils.is_valid_score(1000))

**Q14** Confirm a `@staticmethod` can ALSO be called on an INSTANCE (not just the class) — create a `CreditUtils()` instance and call `.is_valid_score(720)` on it too, showing both ways work identically.

In [ ]:
# YOUR CODE HERE


**Solution 14**

In [ ]:
class CreditUtils:
    @staticmethod
    def is_valid_score(score):
        return 300 <= score <= 850

utils = CreditUtils()
print(utils.is_valid_score(720))
print(CreditUtils.is_valid_score(720))

**Q15** Explain in a comment WHY you would use `@staticmethod` instead of just writing a regular standalone function outside any class — what does grouping it inside the class communicate?

In [ ]:
# YOUR CODE HERE


**Solution 15**

In [ ]:
# A @staticmethod doesn't need self or any instance data — functionally it
# behaves like a plain function. The reason to put it INSIDE a class anyway
# is ORGANIZATION: it signals "this utility function is conceptually related
# to this class" and keeps related logic namespaced together
# (CreditUtils.is_valid_score reads clearly, vs a loose is_valid_score()
# floating in the module that could be about anything).
print("See comment above")

**Q16** Add ANOTHER `@staticmethod` `classify(score)` to `CreditUtils` that returns "Low"/"Medium"/"High" risk (reuse earlier logic). Have it INTERNALLY call `CreditUtils.is_valid_score(score)` first, raising `ValueError` if invalid, before classifying. Test with both a valid and invalid score.

In [ ]:
# YOUR CODE HERE


**Solution 16**

In [ ]:
class CreditUtils:
    @staticmethod
    def is_valid_score(score):
        return 300 <= score <= 850

    @staticmethod
    def classify(score):
        if not CreditUtils.is_valid_score(score):
            raise ValueError("Invalid score")
        if score >= 750:
            return "Low"
        elif score >= 650:
            return "Medium"
        return "High"

print(CreditUtils.classify(720))
try:
    CreditUtils.classify(1000)
except ValueError as e:
    print("Error:", e)

**Q17** Write a class `Money` with a regular instance method `__init__(self, amount)` and a `@staticmethod` method `from_cents(cents)` that returns a NEW `Money` object built from a cents value (`Money(cents / 100)`) — this "alternate constructor" pattern is a very common real use of static methods. Test creating a `Money` object from `4550` cents.

In [ ]:
# YOUR CODE HERE


**Solution 17**

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    @staticmethod
    def from_cents(cents):
        return Money(cents / 100)

m = Money.from_cents(4550)
print(m.amount)

**Q18** Compare: try calling `Money.__init__` behavior via `Money(45.50)` directly (normal constructor) VERSUS `Money.from_cents(4550)` (static alternate constructor) — print both `.amount` values and confirm they produce equivalent objects via two different entry points.

In [ ]:
# YOUR CODE HERE


**Solution 18**

In [ ]:
class Money:
    def __init__(self, amount):
        self.amount = amount

    @staticmethod
    def from_cents(cents):
        return Money(cents / 100)

m1 = Money(45.50)
m2 = Money.from_cents(4550)
print(m1.amount, m2.amount)
print(m1.amount == m2.amount)

## Part E: `@classmethod` — The Other "Class-Level" Tool (Q19-Q22)

**Q19** Define a class `Customer` with a CLASS attribute `total_created = 0`, and a `@classmethod` `get_total(cls)` that returns `cls.total_created` (notice `cls`, not `self` — it receives the CLASS itself, not an instance). Increment `total_created` in `__init__`, create 3 instances, then call `Customer.get_total()`.

In [ ]:
# YOUR CODE HERE


**Solution 19**

In [ ]:
class Customer:
    total_created = 0

    def __init__(self, name):
        self.name = name
        Customer.total_created += 1

    @classmethod
    def get_total(cls):
        return cls.total_created

c1 = Customer("Alice")
c2 = Customer("Bob")
c3 = Customer("Cara")

print(Customer.get_total())

**Q20** Write a `@classmethod` `from_string(cls, data)` on a `Customer` class that parses a string like `"Alice,720"` (splitting on comma) and returns `cls(name, int(score))` — another common "alternate constructor" pattern, this time needing access to the CLASS to build the right type of object. Test it.

In [ ]:
# YOUR CODE HERE


**Solution 20**

In [ ]:
class Customer:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    @classmethod
    def from_string(cls, data):
        name, score = data.split(",")
        return cls(name, int(score))

c = Customer.from_string("Alice,720")
print(c.name, c.score)

**Q21** Explain in a comment the KEY DIFFERENCE between `@staticmethod` and `@classmethod`: what extra thing does a classmethod automatically receive that a staticmethod does not, and why would that matter?

In [ ]:
# YOUR CODE HERE


**Solution 21**

In [ ]:
# @staticmethod receives NOTHING automatically — no self, no cls. It's just
# a function that happens to live inside the class namespace.
#
# @classmethod automatically receives `cls` (the class itself) as its first
# argument. This matters when the method needs to CREATE or reference the
# class itself — e.g. an alternate constructor (like from_string) needs
# to call cls(...) to build a new instance, and this way it automatically
# works correctly even if a SUBCLASS calls it (cls would be the subclass,
# not hardcoded to the parent) — a benefit that becomes clearer once you
# cover inheritance.
print("See comment above")

**Q22** Build a `@classmethod` `default_customer(cls)` on `Customer` that returns a pre-filled default instance: `cls("Unknown", 650)`. Call `Customer.default_customer()` and print the resulting name/score.

In [ ]:
# YOUR CODE HERE


**Solution 22**

In [ ]:
class Customer:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    @classmethod
    def default_customer(cls):
        return cls("Unknown", 650)

c = Customer.default_customer()
print(c.name, c.score)

## Part F: Mini Challenge (Q23-Q25)

**Q23** Build a complete `Account` class combining everything: `self.__balance` (name-mangled private), a `balance` `@property` with getter and a validating setter (rejects negative), a `@staticmethod` `is_valid_amount(amount)` (returns `amount > 0`), and a `deposit(self, amount)` method that uses the static method to validate before adding to `__balance`. Test a valid and invalid deposit.

In [ ]:
# YOUR CODE HERE


**Solution 23**

In [ ]:
class Account:
    def __init__(self, balance=0):
        self.__balance = balance

    @property
    def balance(self):
        return self.__balance

    @balance.setter
    def balance(self, amount):
        if amount >= 0:
            self.__balance = amount

    @staticmethod
    def is_valid_amount(amount):
        return amount > 0

    def deposit(self, amount):
        if not Account.is_valid_amount(amount):
            raise ValueError("Invalid deposit amount")
        self.__balance += amount

acc = Account(1000)
acc.deposit(500)
print(acc.balance)

try:
    acc.deposit(-50)
except ValueError as e:
    print("Error:", e)

**Q24** Add a `@classmethod` `total_accounts(cls)` to the `Account` class from Q23, tracking a class-level counter `accounts_created` incremented in `__init__`. Create 3 accounts and call `Account.total_accounts()`.

In [ ]:
# YOUR CODE HERE


**Solution 24**

In [ ]:
class Account:
    accounts_created = 0

    def __init__(self, balance=0):
        self.__balance = balance
        Account.accounts_created += 1

    @property
    def balance(self):
        return self.__balance

    @classmethod
    def total_accounts(cls):
        return cls.accounts_created

a1 = Account(1000)
a2 = Account(2000)
a3 = Account(500)

print(Account.total_accounts())

**Q25** The big one: build a `CreditCardAccount` class with `self.__balance`, `self.__limit` (both private via name mangling), a `balance` property (read-only — getter only, NO setter, so external code literally cannot assign to it), a `@staticmethod` `calculate_utilization(balance, limit)` returning `balance/limit`, a regular method `charge(self, amount)` that raises `ValueError` if the charge would push utilization above 1.0 (100%), otherwise applies it. Test a charge that succeeds and one that gets rejected for exceeding the limit.

In [ ]:
# YOUR CODE HERE


**Solution 25**

In [ ]:
class CreditCardAccount:
    def __init__(self, limit, balance=0):
        self.__limit = limit
        self.__balance = balance

    @property
    def balance(self):
        return self.__balance
    # No setter defined — balance is read-only from outside the class entirely.

    @staticmethod
    def calculate_utilization(balance, limit):
        return balance / limit

    def charge(self, amount):
        new_balance = self.__balance + amount
        utilization = CreditCardAccount.calculate_utilization(new_balance, self.__limit)
        if utilization > 1.0:
            raise ValueError("Charge would exceed credit limit")
        self.__balance = new_balance

card = CreditCardAccount(limit=5000, balance=1000)
card.charge(2000)
print(card.balance)

try:
    card.charge(3000)
except ValueError as e:
    print("Error:", e)

---
## Checkpoint

25 questions across public/protected/private conventions, `@property` getters/setters, name mangling, `@staticmethod`, `@classmethod`, and mini challenges combining all of it into realistic account classes.

**The one thing to remember from this whole notebook:** Python encapsulation is about COMMUNICATING INTENT (this is internal, don't touch) and giving yourself CONTROL POINTS to validate changes (`@property` setters, methods like `deposit`/`withdraw`) — it is NOT a hard security wall like in some other languages. That's a deliberate design choice in Python ("we're all consenting adults here").

Next up: **Inheritance & Polymorphism** — where `cls` in classmethods and the `__` mangling behavior will make a lot more sense once you see subclasses in action.